In [2]:
import pandas as pd

# 파일 경로 (아까 확인한 경로)
# text_chunking_v3_pipeline 으로 만든 청킹 csv 파일 사용
file_path = './bid_master_chunked_v2.csv'

# 데이터 불러오기
df_new = pd.read_csv(file_path)

# 1. 모든 컬럼명 리스트 출력
print("🔎 현재 CSV의 컬럼 목록:")
print(df_new.columns.tolist())

print("\n" + "-"*30 + "\n")

# 2. 데이터 구성 요약 확인 (결측치나 데이터 타입 확인)
print("📊 데이터 요약 정보:")
df_new.info()

🔎 현재 CSV의 컬럼 목록:
['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일', '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '청크_텍스트', '청크_길이']

------------------------------

📊 데이터 요약 정보:
<class 'pandas.DataFrame'>
RangeIndex: 965 entries, 0 to 964
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      965 non-null    str    
 1   공고 차수      655 non-null    float64
 2   사업명        965 non-null    str    
 3   사업 금액      965 non-null    float64
 4   발주 기관      965 non-null    str    
 5   공개 일자      965 non-null    str    
 6   입찰 참여 시작일  965 non-null    str    
 7   입찰 참여 마감일  800 non-null    str    
 8   사업 요약      965 non-null    str    
 9   파일형식       965 non-null    str    
 10  파일명        965 non-null    str    
 11  청크_텍스트     965 non-null    str    
 12  청크_길이      965 non-null    int64  
dtypes: float64(2), int64(1), str(10)
memory usage: 98.1 KB


In [4]:
# API 키 테스트 코드
import os
from dotenv import load_dotenv
from openai import OpenAI

# 1. .env 파일에 적힌 비밀번호들을 불러옵니다.
load_dotenv()

# 2. 시스템 환경 변수에서 'OPENAI_API_KEY'라는 이름의 값을 꺼내옵니다.
# (이때 .env 파일에 적은 왼쪽 이름과 똑같아야 합니다!)
MY_API_KEY = os.getenv("OPENAI_API_KEY") # 본인 .env 파일에 적은 변수명으로 수정하세요.

# 3. 이제 안전하게 불러온 키를 사용해 클라이언트를 만듭니다.
client = OpenAI(api_key=MY_API_KEY)

try:
    # 아주 짧은 단어 하나만 임베딩 시도
    response = client.embeddings.create(input=["hi"], model="text-embedding-3-small")
    print("✅ API 키 인증 성공!")
except Exception as e:
    print(f"❌ API 키 오류: {e}")
    print("팁: 카드 결제 정보나 잔액(Credit)이 있는지 확인해 보세요.")

✅ API 키 인증 성공!


In [5]:
import pandas as pd
import numpy as np
from openai import OpenAI
from tqdm import tqdm

# 1. 설정
EMBEDDING_MODEL = "text-embedding-3-small"

# 2. 데이터 로드
# text_chunking_v3_pipeline 으로 만든 청킹 csv 파일 사용
file_path = './bid_master_chunked_v2.csv'
df = pd.read_csv(file_path)

# 3. [중요] 컬럼명 통일 작업 (챗봇 코드와 맞추기)
# '공고 번호' -> '공고번호', '텍스트' -> '청크_텍스트'
df.rename(columns={
    '공고 번호': '공고번호',
    '텍스트': '청크_텍스트'
}, inplace=True)

print("✅ 컬럼명 변경 완료:", df.columns.tolist())

# 4. 임베딩 함수 정의
def get_embedding(text):
    if not text or pd.isna(text):
        return None
    text = str(text).replace("\n", " ")
    return client.embeddings.create(input=[text], model=EMBEDDING_MODEL).data[0].embedding

# 5. 임베딩 실행
print(f"🚀 총 {len(df)}건의 데이터를 임베딩합니다. (모델: {EMBEDDING_MODEL})")
tqdm.pandas()
df['embedding'] = df['청크_텍스트'].progress_apply(get_embedding)

# 6. 결과 저장
output_path = './bid_master_optimized_v2.pkl'
df.to_pickle(output_path)

print(f"✨ 모든 작업이 완료되었습니다! 파일 저장 위치: {output_path}")

✅ 컬럼명 변경 완료: ['공고번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일', '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '청크_텍스트', '청크_길이']
🚀 총 965건의 데이터를 임베딩합니다. (모델: text-embedding-3-small)


100%|██████████| 965/965 [03:05<00:00,  5.22it/s]


✨ 모든 작업이 완료되었습니다! 파일 저장 위치: ./bid_master_optimized_v2.pkl
